# Linear system diagnostics

## Usage
LinearSystemDiagnostics.Solve(A,b) returns the solution, the relative residual in the infinity norm, the normwise backward error, a conditioning estimate and its reciprocal. The existing LU Solve is unchanged. All operations are in T.

The estimate uses n additional solves for ||A⁻¹||∞: a non-negligible cost, intended for small matrices. The IsIllConditioned flag is raised if rcond ≤ sqrt(epsilon), as a warning of possible loss of half the digits; it does not certify the forward error or the precision the application requires.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System;
using System.Linq;
using System.Numerics;
using System.Globalization;
using MathemaSharp.LinearAlgebra;
var simple=LinearSystemDiagnostics.Solve<double>(new double[,] {{2,0},{0,4}},[2,8]);
if(simple.ConditionInfinityEstimate!=2 || simple.IsIllConditioned) throw new Exception("Simple condition");
Console.WriteLine("simple_condition: "+simple.ConditionInfinityEstimate);
void Show<T>(int n) where T:IFloatingPointIeee754<T>
{
    var a=new T[n,n];
    var b=new T[n];
    for(var i=0;i<n;i++) for(var j=0;j<n;j++){a[i,j]=T.One/T.CreateChecked(i+j+1);b[i]+=a[i,j];}
    var report=LinearSystemDiagnostics.Solve(a,b);
    Console.WriteLine(typeof(T).Name+"_H"+n+"_condition: "+report.ConditionInfinityEstimate.ToString(null,CultureInfo.InvariantCulture));
    Console.WriteLine(typeof(T).Name+"_H"+n+"_relative_residual: "+report.RelativeResidualInfinity.ToString(null,CultureInfo.InvariantCulture));
    Console.WriteLine(typeof(T).Name+"_H"+n+"_backward_error: "+report.NormwiseBackwardErrorInfinity.ToString(null,CultureInfo.InvariantCulture));
    Console.WriteLine(typeof(T).Name+"_H"+n+"_warning: "+report.IsIllConditioned);
    if(typeof(T)==typeof(double)&&n>=8&&!report.IsIllConditioned) throw new Exception("Expected condition warning");
}
Show<double>(8);
Show<double>(12);
Show<Decimal128>(8);
Console.WriteLine("diagnostics: OK");


simple_condition: 2
Double_H8_condition: 33872790724.675705
Double_H8_relative_residual: 5.015779793873689E-17
Double_H8_backward_error: 2.5078893717486784E-17
Double_H12_condition: 38317455251296850
Double_H12_relative_residual: 1.1762584849436458E-16
Double_H12_backward_error: 5.1053475865956715E-17
Decimal128_H8_condition: 33872791095.00000000000000203105130
Decimal128_H8_relative_residual: 2.074215213899839392612060154840294E-34
Decimal128_H8_backward_error: 1.037107606949919696306030067474925E-34
diagnostics: OK

## Reference and interpretation
- RelativeResidualInfinity = ||Ax−b||∞ / ||b||∞.
- NormwiseBackwardErrorInfinity (alias NormwiseBackwardError) = ||Ax−b||∞ / (||A||∞||x||∞ + ||b||∞).
- ConditionInfinityEstimate ≈ ||A||∞||A⁻¹||∞; ReciprocalConditionInfinityEstimate is its reciprocal.

Residuals computed with scaling and FMA in T, not at higher precision: they can differ from a direct sum and remain subject to rounding/underflow. A false flag does not mean an accurate solution for any tolerance. The estimate depends on the quality of the factorisation; it can become infinite. No refinement or rigorous bound on the forward error. For ill-conditioned original data use oracles and perturbation analysis.
[API Reference](12-api-reference.ipynb), [Hilbert counterexample with oracle](../TimeSeries/13-independent-oracles.ipynb), [architecture](../../ARCHITECTURE.md).
